In [1]:
!pip install --upgrade unsloth unsloth_zoo
!pip install -U torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.7/311.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.8/184.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive', force_remount=True)
FOLDERNAME = 'Colab Notebooks'
%cd drive/MyDrive/$FOLDERNAME/
train_df = pd.read_csv(f"Peter/data/train_data.csv")
test_df = pd.read_csv(f"Peter/data/test_data.csv")

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks


In [3]:
from unsloth import FastLanguageModel
import torch
import os
from datasets import load_dataset, Dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, LlamaForSequenceClassification
from trl import SFTConfig, SFTTrainer
import numpy as np

# Model Name
# model_path = "unsloth/Qwen2.5-0.5B-Instruct"
model_path = "unsloth/Qwen2.5-32B-Instruct"

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.8.9: Fast Qwen2 patching. Transformers: 4.55.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2025.8.9 patched 64 layers with 64 QKV layers, 64 O layers and 64 MLP layers.


In [5]:
seed = 52
device = "cuda" if torch.cuda.is_available() else "cpu"

prompt = '''You are given a comment on reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''

In [6]:
from datasets import load_dataset, Dataset
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from unsloth.chat_templates import standardize_sharegpt

# tokenizer = get_chat_template(
#     tokenizer,
#     chat_template = "qwen-2.5",
# )
user_prompt = """
Subreddit: r/{subreddit}
Rule: {rule}
Examples:
1) {positive_example_1}
Violation: Yes

2) {negative_example_1}
Violation: No

3) {negative_example_2}
Violation: No

4) {positive_example_2}
Violation: Yes
Comment:
{body}
Violation: """

classes = ['No', 'Yes']

columns = ['subreddit', 'rule', 'positive_example_1', 'positive_example_2', 'negative_example_1', 'negative_example_2', 'body', 'rule_violation']
dataset = [
    [{"role": "system", "content": prompt},
    {"role": "user", "content": user_prompt.format(subreddit=subreddit,
                                                   rule=rule,
                                                   positive_example_1=positive_example_1,
                                                   positive_example_2=positive_example_2,
                                                   negative_example_1=negative_example_1,
                                                   negative_example_2=negative_example_2,
                                                   body=body)},
    {"role": "assistant", "content": classes[target]}]
    for subreddit, rule, positive_example_1, positive_example_2, negative_example_1, negative_example_2, body, target  in train_df[columns].values]

def formatting(dataset):
    texts = []
    for i in range(len(dataset)):
        texts.append(tokenizer.apply_chat_template(dataset[i], tokenize=False, add_generation_prompt=False))
    return Dataset.from_dict({'text': texts})

dataset = formatting(dataset)

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 4,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/1929 [00:00<?, ? examples/s]

In [ ]:
# print("檢查 labels 格式:")
# print(type(trainer.train_dataset[0]['labels']))

# trainer.train_dataset = trainer.train_dataset.remove_columns(['labels'])

# print("檢查 labels 格式:")
# print(type(trainer.train_dataset[0]['labels']))

In [8]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map (num_proc=12):   0%|          | 0/1929 [00:00<?, ? examples/s]

In [9]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,929 | Num Epochs = 1 | Total steps = 483
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 268,435,456 of 33,032,311,808 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.170700
20,0.221500
30,0.236900
40,0.231100
50,0.210500
60,0.187500
70,0.222400
80,0.224500
90,0.213400
100,0.188100


In [10]:
from google.colab import userdata
from huggingface_hub import login
HF_key = userdata.get('PLo_HF')
login(token = HF_key)

# model_save = '0824-Qwen2.5-0.5B-Instruct-3E'
model_save = '0827-Qwen2.5-32B-1E'

# model.save_pretrained_merged("0827-Qwen2.5-32B-16bit-1E", tokenizer, save_method = "merged_16bit",)
model.push_to_hub_merged("awilliam60412/0827-Qwen2.5-32B-16bit-1E", tokenizer, save_method = "merged_16bit", token = HF_key)

Unsloth: Saving to awilliam60412/0827-Qwen2.5-32B-16bit-1E will fail, but using a temp folder works! Switching to a temp folder then uploading!


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmphx4eoa4s/tokenizer.json       : 100%|##########| 11.4MB / 11.4MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00014.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00014-of-00014.safetensors:   0%|          | 0.00/2.12G [00:00<?, ?B/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...4s/model-00014-of-00014.safetensors:   1%|1         | 25.1MB / 2.12GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 14/14 [25:48<00:00, 110.58s/it]


In [ ]:
model.push_to_hub_merged("elliefeng25/0827-Qwen2.5-32B-16bit-1E", tokenizer, save_method = "merged_16bit", token = HF_key)